# Build and compare WAHARP on a custom dataset

This notebook loads a two-dimensional point or rectangle dataset, builds the frozen WAHARP policy, and compares it with STR, TGS, Guttman, and R*-tree baselines. It uses one shared generated workload and checks query correctness across methods.

Supported input formats:

- `.npy` with shape `N x 2` (`x,y`) or `N x 4` (`xmin,ymin,xmax,ymax`).
- `.csv` with `x,y`, longitude/latitude aliases, or `xmin,ymin,xmax,ymax` aliases.
- Leave `DATA_PATH = None` to run the included synthetic demonstration.

WAHARP is a static 2D bulk loader. This notebook measures logical node accesses, not physical I/O.

In [ ]:
# Clone the release when running from Colab or an otherwise empty directory.
import os
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/Rifat-Hossain49/Neural_Partitioning.git"
if not Path("realtrain").is_dir():
    checkout = Path("Neural_Partitioning")
    if not checkout.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(checkout)], check=True)
    os.chdir(checkout)

subprocess.run([os.sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository root:", Path.cwd())

## Configuration

For a quick run, keep `WAHARP,STR,TGS`. Guttman and R*-tree use incremental insertion and can be much slower on large datasets. Set `MAX_OBJECTS` to a positive number for a deterministic sample while testing the notebook.

In [ ]:
DATA_PATH = None                 # Example: '/content/my_rectangles.npy'
OUTPUT_DIRECTORY = "custom_benchmark_output"
CAPACITY = 128
METHODS = "WAHARP,STR,TGS"    # Add Guttman,RStar when desired
MAX_OBJECTS = 0                  # 0 uses every input object
SYNTHETIC_OBJECTS = 10_000       # Used only when DATA_PATH is None
CONSTRUCTION_QUERIES = 512
RANGE_QUERIES_PER_SCALE = 100   # Three range-window scales
POINT_QUERIES = 100
KNN_QUERIES = 100
KNN_K = [1, 10, 100]
SEED = 20260927

In [ ]:
command = [
    os.sys.executable,
    "scripts/custom_dataset_benchmark.py",
    "--output", OUTPUT_DIRECTORY,
    "--capacity", str(CAPACITY),
    "--methods", METHODS,
    "--max-objects", str(MAX_OBJECTS),
    "--synthetic-objects", str(SYNTHETIC_OBJECTS),
    "--construction-queries", str(CONSTRUCTION_QUERIES),
    "--range-queries", str(RANGE_QUERIES_PER_SCALE),
    "--point-queries", str(POINT_QUERIES),
    "--knn-queries", str(KNN_QUERIES),
    "--knn-k", *[str(value) for value in KNN_K],
    "--seed", str(SEED),
]
if DATA_PATH is not None:
    command.extend(["--data", str(DATA_PATH)])

print("Running:", " ".join(command))
subprocess.run(command, check=True)

## Results

`ratio_to_waharp < 1` means the baseline used fewer logical node accesses; a value above one means WAHARP used fewer. Every method must have zero correctness mismatches before its comparison is accepted.

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(Path(OUTPUT_DIRECTORY) / "SUMMARY.csv")
families = pd.read_csv(Path(OUTPUT_DIRECTORY) / "BY_QUERY_FAMILY.csv")
display(summary)
display(families)

assert summary["tree_valid"].all(), "At least one constructed tree is invalid"
assert (summary["correctness_mismatches_vs_reference"] == 0).all(), "Correctness mismatch detected"
print("All structural and query-correctness checks passed.")

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(summary["method"], summary["total_node_accesses"], color="#287271")
axes[0].set_title("Logical node accesses")
axes[0].set_ylabel("Total accesses (lower is better)")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(summary["method"], summary["build_seconds"], color="#D17A22")
axes[1].set_title("Construction time")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=25)
figure.tight_layout()
figure.savefig(Path(OUTPUT_DIRECTORY) / "COMPARISON.png", dpi=180)
plt.show()

## About PLATON

The exact PLATON comparison in the paper uses the authors' implementation, its patched native `libspatialindex` build, and fixed MCTS settings. That third-party source bundle is not redistributed in this repository. The recorded PLATON experiment remains reproducible through `scripts/kaggle/run_fullscale.py` and the protocol described in `docs/REPRODUCING.md` once the author support bundle is supplied. Do not compare against a reimplementation and label it as the paper's PLATON baseline.